In [1]:
import requests
import json


In [2]:
import requests
import xml.etree.ElementTree as ET

# GetCapabilities - pobranie informacji o dostępnych warstwach
print("=== Pobieranie GetCapabilities ===")
capabilities_url = "https://mapy.geoportal.gov.pl/wss/service/rcn?SERVICE=WMS&REQUEST=GetCapabilities"
response = requests.get(capabilities_url)

# Sprawdzenie statusu odpowiedzi
if response.status_code == 200:
    print(f"Status: {response.status_code} - OK")
    print(f"Długość odpowiedzi: {len(response.text)} znaków")
    print(f"\nPierwsze 1000 znaków:\n{response.text[:1000]}")
    
    # Próba parsowania XML
    try:
        root = ET.fromstring(response.text)
        print(f"\nTag główny: {root.tag}")
        
        # Wyodrębnienie dostępnych warstw
        for layer in root.findall('.//{http://www.opengis.net/wms}Layer'):
            name = layer.find('{http://www.opengis.net/wms}Name')
            title = layer.find('{http://www.opengis.net/wms}Title')
            if name is not None and name.text:
                print(f"  Layer: {name.text}")
                if title is not None:
                    print(f"    Tytuł: {title.text}")
    except Exception as e:
        print(f"Błąd parsowania XML: {e}")
else:
    print(f"Błąd: Status {response.status_code}")


=== Pobieranie GetCapabilities ===
Status: 200 - OK
Długość odpowiedzi: 14463 znaków

Pierwsze 1000 znaków:
<?xml version='1.0' encoding="UTF-8" standalone="no" ?>
<WMS_Capabilities version="1.3.0"  xmlns="http://www.opengis.net/wms"   xmlns:sld="http://www.opengis.net/sld"   xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"   xmlns:ms="http://mapserver.gis.umn.edu/mapserver"   xmlns:inspire_common="http://inspire.ec.europa.eu/schemas/common/1.0"   xmlns:inspire_vs="http://inspire.ec.europa.eu/schemas/inspire_vs/1.0"   xsi:schemaLocation="http://www.opengis.net/wms http://schemas.opengis.net/wms/1.3.0/capabilities_1_3_0.xsd  http://www.opengis.net/sld http://schemas.opengis.net/sld/1.1.0/sld_capabilities.xsd  http://inspire.ec.europa.eu/schemas/inspire_vs/1.0  http://inspire.ec.europa.eu/schemas/inspire_vs/1.0/inspire_vs.xsd http://mapserver.gis.umn.edu/mapserver https://mapy.geoportal.gov.pl/wss/service/rcn?language=pol&amp;service=WMS&amp;version=1.3.0&amp;request=GetSchemaExtens

In [3]:

# Współrzędne Lublina (EPSG:2180 - układ polski)
# Lublin centralne: około 723000, 688000 (EPSG:2180)
lublin_bbox = "723000,687000,724000,689000"  # minX, minY, maxX, maxY

print("=== Pobieranie GetMap - Budynki ===")
get_map_url = (
    "https://mapy.geoportal.gov.pl/wss/service/rcn?"
    "SERVICE=WMS&REQUEST=GetMap&LAYERS=budynki"
    "&BBOX=723000,687000,724000,689000&WIDTH=400&HEIGHT=400"
    "&FORMAT=image/png&SRS=EPSG:2180&VERSION=1.3.0"
)

map_response = requests.get(get_map_url)
if map_response.status_code == 200:
    with open("/tmp/lublin_map.png", "wb") as f:
        f.write(map_response.content)
    print(f"Mapa zapisana, rozmiar: {len(map_response.content)} bajtów")
else:
    print(f"Błąd: Status {map_response.status_code}")


=== Pobieranie GetMap - Budynki ===
Mapa zapisana, rozmiar: 460 bajtów


In [4]:

# GetFeatureInfo dla lokalów (mieszkań)
print("=== GetFeatureInfo - Lokale (Mieszkania) ===")

# Punkt centralny Lublina w EPSG:2180
x, y = 723500, 688000

get_feature_url = (
    "https://mapy.geoportal.gov.pl/wss/service/rcn?"
    "SERVICE=WMS&REQUEST=GetFeatureInfo&LAYERS=lokale"
    f"&X={x}&Y={y}&WIDTH=400&HEIGHT=400"
    "&BBOX=723000,687000,724000,689000"
    "&CRS=EPSG:2180&VERSION=1.3.0&STYLES="
    "&QUERY_LAYERS=lokale&INFO_FORMAT=application/json"
)

feature_response = requests.get(get_feature_url)
print(f"Status: {feature_response.status_code}")
print(f"Typ zawartości: {feature_response.headers.get('content-type', 'unknown')}")
print(f"\nOdpowiedź (pierwsze 3000 znaków):\n{feature_response.text[:3000]}")

# Próba parsowania JSON
if feature_response.text.strip():  # Sprawdzenie czy odpowiedź nie jest pusta
    try:
        data = feature_response.json()
        print(f"\n✓ Udało się sparsować JSON")
        print(f"Struktura danych: {type(data)}")
        if isinstance(data, dict):
            print(f"Klucze: {list(data.keys())}")
            # Wydrukowanie zawartości
            import json
            print(json.dumps(data, indent=2, ensure_ascii=False)[:2000])
    except Exception as e:
        print(f"\n✗ Błąd parsowania JSON: {e}")
else:
    print("\n✗ Odpowiedź jest pusta")


=== GetFeatureInfo - Lokale (Mieszkania) ===
Status: 200
Typ zawartości: text/xml; charset=UTF-8

Odpowiedź (pierwsze 3000 znaków):
<?xml version='1.0' encoding="UTF-8" standalone="no" ?>
<ServiceExceptionReport version="1.3.0" xmlns="http://www.opengis.net/ogc" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.opengis.net/ogc http://schemas.opengis.net/wms/1.3.0/exceptions_1_3_0.xsd">
<ServiceException code="InvalidPoint">
msWMSFeatureInfo(): WMS server error. Invalid I/J values
</ServiceException>
</ServiceExceptionReport>


✗ Błąd parsowania JSON: Expecting value: line 1 column 1 (char 0)


In [5]:

# Spróbuję innego podejścia - poszukiwanie usługi WFS
print("=== Wyszukiwanie usługi WFS ===")

# Typowe adresy usług WFS dla GUGiK
wfs_base_urls = [
    "https://mapy.geoportal.gov.pl/wss/service/rcn",
    "https://services.arcgisonline.com/arcgis/rest/services",
]

# Spróbujmy GetCapabilities z SERVICE=WFS
capabilities_wfs_url = "https://mapy.geoportal.gov.pl/wss/service/rcn?SERVICE=WFS&REQUEST=GetCapabilities&VERSION=2.0.0"
response_wfs = requests.get(capabilities_wfs_url, timeout=10)

print(f"Status WFS: {response_wfs.status_code}")
if response_wfs.status_code == 200:
    print(f"Typ zawartości: {response_wfs.headers.get('content-type', 'unknown')}")
    print(f"Długość odpowiedzi: {len(response_wfs.text)} znaków")
    print(f"\nPierwsze 1500 znaków:\n{response_wfs.text[:1500]}")
else:
    print(f"Błąd: WFS niedostępny")


=== Wyszukiwanie usługi WFS ===
Status WFS: 200
Typ zawartości: text/xml; charset=UTF-8
Długość odpowiedzi: 19574 znaków

Pierwsze 1500 znaków:
<?xml version="1.0" encoding="UTF-8"?>
<wfs:WFS_Capabilities xmlns:gml="http://www.opengis.net/gml/3.2" xmlns:wfs="http://www.opengis.net/wfs/2.0" xmlns:ows="http://www.opengis.net/ows/1.1" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:fes="http://www.opengis.net/fes/2.0" xmlns:ms="http://mapserver.gis.umn.edu/mapserver" xmlns:inspire_common="http://inspire.ec.europa.eu/schemas/common/1.0" xmlns:inspire_dls="http://inspire.ec.europa.eu/schemas/inspire_dls/1.0" xmlns="http://www.opengis.net/wfs/2.0" version="2.0.0" xsi:schemaLocation="http://www.opengis.net/wfs/2.0 http://schemas.opengis.net/wfs/2.0/wfs.xsd http://inspire.ec.europa.eu/schemas/inspire_dls/1.0 http://inspire.ec.europa.eu/schemas/inspire_dls/1.0/inspire_dls.xsd http://inspire.ec.europa.eu/schemas/common/1.0 http://inspire.ec.

In [6]:

# Parsowanie WFS GetCapabilities
print("=== Dostępne typy obiektów (Feature Types) ===")

root_wfs = ET.fromstring(response_wfs.text)

# Przestrzenie nazw
ns = {
    'wfs': 'http://www.opengis.net/wfs/2.0',
    'ows': 'http://www.opengis.net/ows/1.1'
}

# Znalezienie Feature Types
feature_types = root_wfs.findall('.//wfs:FeatureType', ns)
print(f"Liczba dostępnych typów obiektów: {len(feature_types)}\n")

for ft in feature_types:
    name_elem = ft.find('wfs:Name', ns)
    title_elem = ft.find('wfs:Title', ns)
    abstract_elem = ft.find('wfs:Abstract', ns)
    
    if name_elem is not None:
        name = name_elem.text
        title = title_elem.text if title_elem is not None else "---"
        abstract = abstract_elem.text if abstract_elem is not None else "---"
        
        print(f"Nazwa: {name}")
        print(f"  Tytuł: {title}")
        print(f"  Opis: {abstract[:100]}...")
        print()


=== Dostępne typy obiektów (Feature Types) ===
Liczba dostępnych typów obiektów: 4

Nazwa: ms:budynki
  Tytuł: budynki
  Opis: ---...

Nazwa: ms:lokale
  Tytuł: lokale
  Opis: ---...

Nazwa: ms:dzialki
  Tytuł: działki
  Opis: ---...

Nazwa: ms:powiaty
  Tytuł: powiaty
  Opis: ---...



In [7]:

# Pobranie danych lokalów dla Lublina przy użyciu WFS GetFeature
print("=== Pobranie danych lokalów (mieszkań) dla Lublina ===\n")

# Spróbujemy z różnymi formatami
formats_to_try = [
    "gml/3.2.1",
    "text/xml",
    "gml32",
    "gml",
]

response_data = None
for fmt in formats_to_try:
    # WFS GetFeature zapytanie dla lokalów
    wfs_url = (
        "https://mapy.geoportal.gov.pl/wss/service/rcn?"
        "SERVICE=WFS&VERSION=2.0.0&REQUEST=GetFeature"
        "&TYPENAME=ms:lokale"
        "&BBOX=723000,687000,724000,689000,urn:ogc:def:crs:EPSG:2180"
        f"&outputformat={fmt}"
    )
    
    print(f"Próba z formatem: {fmt}")
    response_data = requests.get(wfs_url, timeout=15)
    print(f"  Status: {response_data.status_code}")
    
    if response_data.status_code == 200:
        print(f"  ✓ Sukces!")
        break
    else:
        print(f"  Błąd: {response_data.text[:200]}")

if response_data and response_data.status_code == 200:
    print(f"\nTyp zawartości: {response_data.headers.get('content-type', 'unknown')}")
    print(f"Rozmiar odpowiedzi: {len(response_data.text)} znaków\n")
    
    # Parsowanie GML/XML
    try:
        root = ET.fromstring(response_data.text)
        print(f"✓ Udało się sparsować XML")
        print(f"Tag główny: {root.tag}")
        print(f"\nPierwsze 2000 znaków:")
        print(response_data.text[:2000])
        
        # Liczenie Features
        features = root.findall('.//{http://www.opengis.net/gml/3.2}featureMember')
        print(f"\nLiczba znalezionych Features: {len(features)}")
        
    except Exception as e:
        print(f"✗ Błąd parsowania XML: {e}")
        print(f"Zawartość: {response_data.text[:1000]}")


=== Pobranie danych lokalów (mieszkań) dla Lublina ===

Próba z formatem: gml/3.2.1


ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [8]:

# Spróbuję z większym zakresem i bez ograniczenia BBOX
# print("=== Pobranie danych lokalów bez ograniczenia BBOX ===\n")

# # Najpierw sprawdzę opisFeatureType (DescribeFeatureType)
# describe_url = (
#     "https://mapy.geoportal.gov.pl/wss/service/rcn?"
#     "SERVICE=WFS&VERSION=2.0.0&REQUEST=DescribeFeatureType"
#     "&TYPENAME=ms:lokale&OUTPUTFORMAT=application/json"
# )

# print("Pobranie schematu danych (DescribeFeatureType)...")
# response_describe = requests.get(describe_url, timeout=15)
# print(f"Status: {response_describe.status_code}")

# if response_describe.status_code == 200:
#     print(response_describe.text[:2000])
    
# Teraz spróbuję bez BBOX - pobrać dane z filtrem
print("\n=== Pobranie lokalów bez ograniczenia BBOX ===\n")

# Większy zakres dla powiatu lubelskiego (EPSG:2180)
# Pow. lubelski: ~ 710000-730000 X, 680000-700000 Y
wfs_url_full = (
    "https://mapy.geoportal.gov.pl/wss/service/rcn?"
    "SERVICE=WFS&VERSION=2.0.0&REQUEST=GetFeature"
    "&TYPENAME=ms:lokale"
    "&BBOX=373830,741742,387807,754350,urn:ogc:def:crs:EPSG:2180"
    "&outputformat=gml32"
)

print(f"Zapytanie: {wfs_url_full[:120]}...\n")
response_full = requests.get(wfs_url_full, timeout=300)

print(f"Status: {response_full.status_code}")
print(f"Typ zawartości: {response_full.headers.get('content-type', 'unknown')}")

if response_full.status_code == 200:
    root_full = ET.fromstring(response_full.text)
    print(f"Tag główny: {root_full.tag}")
    print(f"numberMatched: {root_full.get('numberMatched')}")
    print(f"numberReturned: {root_full.get('numberReturned')}")
    print(f"\nRozmiar odpowiedzi: {len(response_full.text)} znaków")
    print(f"\nPierwsze 3000 znaków:\n{response_full.text[:3000]}")
else:
    print(f"Błąd:\n{response_full.text[:1000]}")



=== Pobranie lokalów bez ograniczenia BBOX ===

Zapytanie: https://mapy.geoportal.gov.pl/wss/service/rcn?SERVICE=WFS&VERSION=2.0.0&REQUEST=GetFeature&TYPENAME=ms:lokale&BBOX=37383...

Status: 200
Typ zawartości: text/xml; subtype="gml/3.2.1"; charset=UTF-8
Tag główny: {http://www.opengis.net/wfs/2.0}FeatureCollection
numberMatched: 96666
numberReturned: 96666

Rozmiar odpowiedzi: 211028932 znaków

Pierwsze 3000 znaków:
<?xml version='1.0' encoding="UTF-8" ?>
<wfs:FeatureCollection
   xmlns:ms="http://mapserver.gis.umn.edu/mapserver"
   xmlns:gml="http://www.opengis.net/gml/3.2"
   xmlns:wfs="http://www.opengis.net/wfs/2.0"
   xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
   xsi:schemaLocation="http://mapserver.gis.umn.edu/mapserver https://mapy.geoportal.gov.pl/wss/service/rcn?SERVICE=WFS&amp;VERSION=2.0.0&amp;REQUEST=DescribeFeatureType&amp;TYPENAME=ms:lokale&amp;OUTPUTFORMAT=application%2Fgml%2Bxml%3B%20version%3D3.2 http://www.opengis.net/wfs/2.0 http://schemas.opengis.net/

In [9]:

import pandas as pd
import geopandas as gpd
from io import StringIO

print("=== Analiza danych lokalów ===\n")

if response_full.status_code == 200:
    # Zapisanie odpowiedzi do pliku
    with open('/tmp/lokale.gml', 'w', encoding='utf-8') as f:
        f.write(response_full.text)
    
    # Wczytanie GML za pomocą GeoPandas
    try:
        gdf = gpd.read_file('/tmp/lokale.gml')
        print(f"✓ Wczytano dane przy użyciu GeoPandas")
        print(f"Liczba lokalów: {len(gdf)}")
        print(f"\nKolumny: {list(gdf.columns)}")
        print(f"\nPierwsze rekordy:\n{gdf.head()}")
        
        # Statystyki
        print(f"\n=== Statystyki ===")
        print(f"Rodzaje (gdf.dtypes):\n{gdf.dtypes}")
        
    except Exception as e:
        print(f"Błąd przy wczytaniu GeoPandas: {e}")
        
        # Alternatywny sposób - parsowanie ręczne
        print(f"\nPróba ręcznego parsowania danych...")
        root = ET.fromstring(response_full.text)
        
        # Znalezienie wszystkich Features
        ns = {
            'gml': 'http://www.opengis.net/gml/3.2',
            'wfs': 'http://www.opengis.net/wfs/2.0',
            'ms': 'http://mapserver.gis.umn.edu/mapserver'
        }
        
        features = root.findall('.//wfs:member', ns)
        print(f"Liczba Features: {len(features)}")
        
        # Wyodrębnienie danych
        data = []
        for feature in features:
            # Znalezienie wszystkich pól w Feature
            feature_data = {}
            for elem in feature.iter():
                if elem.tag.startswith('{http://mapserver.gis.umn.edu/mapserver}'):
                    tag = elem.tag.replace('{http://mapserver.gis.umn.edu/mapserver}', '')
                    if elem.text:
                        feature_data[tag] = elem.text
            
            if feature_data:
                data.append(feature_data)
        
        if data:
            df = pd.DataFrame(data)
            print(f"\n✓ Wyodrębniono {len(df)} rekordów")
            print(f"Kolumny: {list(df.columns)}")
            print(f"\nPierwsze rekordy:\n{df.head(10)}")
            
            # Zapisanie do CSV
            df.to_csv('/tmp/lokale.csv', index=False, encoding='utf-8')
            print(f"\n✓ Dane zapisane do /tmp/lokale.csv")
else:
    print(f"Błąd pobierania danych: {response_full.status_code}")


=== Analiza danych lokalów ===

✓ Wczytano dane przy użyciu GeoPandas
Liczba lokalów: 96666

Kolumny: ['gml_id', 'serwis_rcn', 'teryt', 'tran_przestrzen_nazw', 'tran_lokalny_id_iip', 'tran_wersja_id', 'tran_rodzaj_trans', 'tran_rodzaj_rynku', 'tran_sprzedajacy', 'tran_kupujacy', 'tran_cena_brutto', 'tran_vat', 'dok_data', 'nier_rodzaj', 'nier_prawo', 'nier_udzial', 'nier_pow_gruntu', 'nier_cena_brutto', 'nier_vat', 'lok_id_lokalu', 'lok_nr_lokalu', 'lok_funkcja', 'lok_liczba_izb', 'lok_nr_kond', 'lok_pow_uzyt', 'lok_pow_przyn', 'lok_cena_brutto', 'lok_vat', 'lok_adres', 'geometry']

Pierwsze rekordy:
           gml_id serwis_rcn teryt tran_przestrzen_nazw  \
0  lokale.7652240       None  0609    PL.PZGiK.9349.RCN   
1  lokale.7702785       None  0663    PL.PZGiK.4884.RCN   
2  lokale.7691559       None  0663    PL.PZGiK.4884.RCN   
3  lokale.7682760       None  0663    PL.PZGiK.4884.RCN   
4  lokale.7693503       None  0663    PL.PZGiK.4884.RCN   

                    tran_lokalny_id_i

In [ ]:

# Analiza cen mieszkań
print("=== Analiza cen mieszkań w Lublinie ===\n")

# Ponowne wczytanie danych
try:
    df_lokale = pd.read_csv('/tmp/lokale.csv')
    
    print(f"Liczba lokalów: {len(df_lokale)}")
    print(f"\nPodstawowe informacje:")
    print(df_lokale.info())
    
    # Szukanie kolumn z cenami
    print(f"\nKolumny zawierające 'cen': {[col for col in df_lokale.columns if 'cen' in col.lower()]}")
    print(f"Kolumny zawierające 'cena': {[col for col in df_lokale.columns if 'cena' in col.lower()]}")
    print(f"Kolumny zawierające 'price': {[col for col in df_lokale.columns if 'price' in col.lower()]}")
    print(f"Kolumny zawierające 'wartość': {[col for col in df_lokale.columns if 'wartość' in col.lower()]}")
    
    # Wyświetlenie wszystkich kolumn
    print(f"\nWszystkie kolumny ({len(df_lokale.columns)}):")
    for col in df_lokale.columns:
        print(f"  - {col}: {df_lokale[col].dtype}")
    
    # Wyświetlenie przykładowych danych
    print(f"\nPierwsze 5 wierszy:")
    print(df_lokale.head())
    
    # Statystyki dla kolumn numerycznych
    print(f"\nStatystyki opisowe:")
    print(df_lokale.describe())
    
except FileNotFoundError:
    print("Plik nie znaleziony. Dane mogą nie być dostępne.")
except Exception as e:
    print(f"Błąd: {e}")


=== Analiza cen mieszkań w Lublinie ===

Plik nie znaleziony. Dane mogą nie być dostępne.


In [ ]:

# Sprawdzenie i ponowne przetworzenie danych
print("=== Sprawdzenie przetworzonych danych ===\n")

if response_full.status_code == 200:
    root = ET.fromstring(response_full.text)
    
    ns = {
        'gml': 'http://www.opengis.net/gml/3.2',
        'wfs': 'http://www.opengis.net/wfs/2.0',
        'ms': 'http://mapserver.gis.umn.edu/mapserver'
    }
    
    # Znalezienie wszystkich elementów ms:lokale
    features = root.findall('.//ms:lokale', ns)
    print(f"Liczba znalezionych lokalów: {len(features)}")
    
    # Wyodrębnienie danych
    data = []
    for idx, feature in enumerate(features):
        feature_data = {}
        
        # Iteruj przez wszystkie dzieci Feature
        for child in feature:
            # Usuń namespace z tagu
            tag = child.tag.split('}')[-1] if '}' in child.tag else child.tag
            
            if child.text:
                feature_data[tag] = child.text
            else:
                # Jeśli brak tekstu, spróbuj znaleźć wartości w podelementach
                for subchild in child:
                    subtag = subchild.tag.split('}')[-1] if '}' in subchild.tag else subchild.tag
                    if subchild.text:
                        feature_data[f"{tag}_{subtag}"] = subchild.text
        
        if feature_data:
            data.append(feature_data)
    
    print(f"Wyodrębniono {len(data)} rekordów")
    
    if data:
        df_lokale = pd.DataFrame(data)
        print(f"\nKolumny ({len(df_lokale.columns)}):")
        for col in sorted(df_lokale.columns):
            non_null = df_lokale[col].notna().sum()
            print(f"  {col}: {non_null} wartości")
        
        print(f"\nPierwsze 3 wiersze:")
        print(df_lokale.head(3).to_string())
        
        # Zapisanie do CSV
        output_path = '/home/dron/Studia/eskploracja/lokale_lublin.csv'
        df_lokale.to_csv(output_path, index=False, encoding='utf-8')
        print(f"\n✓ Dane zapisane do {output_path}")


=== Sprawdzenie przetworzonych danych ===

Liczba znalezionych lokalów: 39
Wyodrębniono 39 rekordów

Kolumny (24):
  boundedBy: 39 wartości
  dok_data: 39 wartości
  lok_funkcja: 39 wartości
  lok_id_lokalu: 39 wartości
  lok_liczba_izb: 39 wartości
  lok_nr_kond: 9 wartości
  lok_nr_lokalu: 39 wartości
  lok_pow_przyn: 3 wartości
  lok_pow_uzyt: 39 wartości
  msGeometry: 39 wartości
  nier_cena_brutto: 28 wartości
  nier_pow_gruntu: 39 wartości
  nier_prawo: 39 wartości
  nier_rodzaj: 39 wartości
  nier_udzial: 39 wartości
  teryt: 39 wartości
  tran_cena_brutto: 39 wartości
  tran_kupujacy: 39 wartości
  tran_lokalny_id_iip: 39 wartości
  tran_przestrzen_nazw: 39 wartości
  tran_rodzaj_rynku: 30 wartości
  tran_rodzaj_trans: 39 wartości
  tran_sprzedajacy: 39 wartości
  tran_wersja_id: 16 wartości

Pierwsze 3 wiersze:
      boundedBy    msGeometry teryt tran_przestrzen_nazw                   tran_lokalny_id_iip tran_rodzaj_trans tran_rodzaj_rynku tran_sprzedajacy  tran_kupujacy tran_

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 2000)
pd.set_option('display.max_rows', 100)

df_lokale.head()

,boundedBy,msGeometry,teryt,tran_przestrzen_nazw,tran_lokalny_id_iip,tran_rodzaj_trans,tran_rodzaj_rynku,tran_sprzedajacy,tran_kupujacy,tran_cena_brutto,dok_data,nier_rodzaj,nier_prawo,nier_udzial,nier_pow_gruntu,nier_cena_brutto,lok_id_lokalu,lok_nr_lokalu,lok_funkcja,lok_liczba_izb,lok_pow_uzyt,tran_wersja_id,lok_pow_przyn,lok_nr_kond,tran_cena_brutto_num,nier_cena_brutto_num,lok_pow_uzyt_num,lok_liczba_izb_num,cena_za_m2
0,\n \t,\n,2818,PL.PZGiK.7563.RCN,A12F967F-94AA-4405-9833-E9C0326644B3,wolnyRynek,wtorny,osobaFizyczna,osobaFizyczna,95000,2024-09-23 02:00:00+02,nieruchomoscLokalowa,wlasnoscLokaluWrazZPrawemZwiazanym,1/1,0.0488,0,281801_2.0001.381_BUD.6_LOK,6_LOK,mieszkalna,4,60.75,NaN,NaN,NaN,95000,0.0,60.75,4,1563.786008
1,\n \t,\n,2819,PL.PZGiK.10911.RCN,F8F853E1-125B-4D4F-8DF3-210C115F949B,wolnyRynek,NaN,osobaFizyczna,osobaFizyczna,39000,2023-08-18 02:00:00+02,nieruchomoscLokalowa,wlasnoscNieruchomosciGruntowej,1/1,275,NaN,281901_2.0018.330/15.1_BUD.4_LOK,4_LOK,mieszkalna,2,35,2023-09-13T14:00:39,NaN,NaN,39000,NaN,35.00,2,1114.285714
2,\n \t,\n,2818,PL.PZGiK.7563.RCN,25C72AAC-5824-4C04-999F-6DE6E4B886C8,wolnyRynek,wtorny,osobaFizyczna,osobaFizyczna,40000,2025-10-10 02:00:00+02,nieruchomoscLokalowa,wlasnoscLokaluWrazZPrawemZwiazanym,1/1,0.1389,0,281801_2.0009.92_BUD.2_LOK,2_LOK,mieszkalna,4,61,NaN,NaN,NaN,40000,0.0,61.00,4,655.737705
3,\n \t,\n,2818,PL.PZGiK.7563.RCN,DDF52C4F-2152-43B5-B554-D5687DBF5CB9,wolnyRynek,wtorny,osobaFizyczna,osobaFizyczna,40000,2025-02-21 01:00:00+01,nieruchomoscLokalowa,wlasnoscLokaluWrazZPrawemZwiazanym,1/1,0.1389,0,281801_2.0009.92_BUD.1_LOK,1_LOK,mieszkalna,4,77.45,NaN,NaN,NaN,40000,0.0,77.45,4,516.462234
4,\n \t,\n,2818,PL.PZGiK.7563.RCN,6BFE6A70-A6D9-4C34-AE22-D3327CC410E3,wolnyRynek,wtorny,osobaFizyczna,osobaFizyczna,120000,2025-08-28 02:00:00+02,nieruchomoscLokalowa,wlasnoscLokaluWrazZPrawemZwiazanym,1/1,0.021,0,281801_2.0013.103_BUD.2_LOK,2_LOK,mieszkalna,3,78.22,NaN,12.11,NaN,120000,0.0,78.22,3,1534.134492
